1. Create a program that will read in all the images from a foler and use them to get the calibration parameters. Calibrate your camera using a minimum of three data sets. Compare and contrast the results. Do they make sense. One of the data sets should have at least 25 images. Calibrate the camera using 2, 5, 10, 15, and 25 images. Compare the results. What do they tell us? 

In [ ]:
# Locate utils whether the notebook opens from homework or the repository root.
from pathlib import Path
import sys

root = Path.cwd() if Path("images").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / "homework"))

from utils.CalibrationComparison import CalibrationComparison

results = CalibrationComparison().compare()


Comparison of the calibration results

Only `filtered_set_me566_1` is used. Of its 816 images, 813 had detectable 7 x 10 inner corners. These are split into three disjoint groups of 271 images using a fixed random order. They are three subsets of one recording, not three independent recordings.

The code uses `findChessboardCorners` to detect corners, `cornerSubPix` to refine them, and `calibrateCamera` to obtain the camera matrix, distortion coefficients, and image rotations/translations. Focal lengths and principal points are in pixels. Square size is set to 1, so translations are in square-width units; replace it with the measured square size for physical units.

Sets 1A, 1B and 1C give horizontal focal lengths of 1765.75, 1764.15 and 1748.98 pixels, respectively, with RMS reprojection errors of 0.7499, 1.1473 and 0.9627 pixels. The focal lengths differ by less than 1%, and k1 stays near -0.51. This agreement makes sense because the images come from the same camera recording. The horizontal and vertical focal lengths are also nearly equal, consistent with square pixels. Higher-order distortion is less stable: k3 ranges from 0.104 to 0.611. Different board poses and corner quality can affect the estimates.

For 2, 5, 10, 15 and 25 images, the horizontal focal lengths are 1642.64, 1665.02, 1737.37, 1798.85 and 1792.43 pixels. The corresponding RMS errors are 0.1659, 0.1963, 0.1892, 0.3045 and 0.8866 pixels. Although the two-image fit has the lowest error, its focal length is about 6.6% below the average of the three large groups. The principal point also changes substantially with small samples: cy is 770.00 pixels with five images, compared with about 603-617 pixels in the large groups. Low fitting error alone therefore does not establish reliable camera parameters.

The focal length changes by only about 0.4% between 15 and 25 images, suggesting improved stability for these particular subsets, but the distortion coefficients still vary. Each larger trial retains the previous images. More varied board angles, distances and positions can constrain calibration better, while many similar frames add less information. Error need not decrease when more difficult images are added. These are training errors from one subset per size, so they do not prove accuracy on new images.

2. Filter the noisy images using a 5x5 gaussian kernel, a 3x3 gaussian kernel and another noise suppressing filter. Compare and contrast the results. Do this with the images listed below: 
- Camera_man_w_noise
- SaltAndPepperNoise
- GaussianNoise 
- UniformNoise
Make sure to answer questions like which image each filter works best for? What metric is used to determine the quality of the filtering? What filter would be recommended for what types of noise? The writeup needs to contain filtered and original images. It also needs to describe the code, what functions were used, inputs to the function, and settings. The selection of setting and such need to be justified in your writeup. 

In [ ]:
# Read the JPGs and display the original beside the filtered images.
from pathlib import Path
import sys
import cv2
import numpy as np

root = Path.cwd() if Path("images").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / "homework"))
from utils.ImageReader import ImageReader
from utils.ShowResults import show_results

image_names = ["camera_man_w_noise.jpg", "SaltAndPepperNoise.jpg",
               "GaussianNoise.jpg", "UniformNoise.jpg"]
for name in image_names:
    image = ImageReader(root / "images" / name, grayscale=True).images[0]
    gaussian3 = cv2.GaussianBlur(image, (3, 3), 0)
    gaussian5 = cv2.GaussianBlur(image, (5, 5), 0)
    median3 = cv2.medianBlur(image, 3)
    show_results([image, gaussian3, gaussian5, median3],
                 ["Original", "Gaussian 3 x 3", "Gaussian 5 x 5", "Median 3 x 3"],
                 Path(name).stem + "_comparison")


3. Repeat the above with a median filter and an n-rank filter. Use both a 3x3 and 5x5 window. 

In [ ]:
# Read the JPGs and display the original beside the filtered images.
from pathlib import Path
import sys
import cv2
import numpy as np

root = Path.cwd() if Path("images").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / "homework"))
from utils.ImageReader import ImageReader
from utils.ShowResults import show_results

def rank_filter(image, size, rank):
    # Rank is zero-based: 0 selects the smallest value.
    padded = np.pad(image, size // 2, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(padded, (size, size))
    neighbors = windows.reshape(*image.shape, size * size)
    return np.partition(neighbors, rank, axis=-1)[..., rank]

image_names = ["camera_man_w_noise.jpg", "SaltAndPepperNoise.jpg",
               "GaussianNoise.jpg", "UniformNoise.jpg"]
for name in image_names:
    image = ImageReader(root / "images" / name, grayscale=True).images[0]
    # Lower-quartile ranks: third of 9 values and seventh of 25 values.
    median3 = cv2.medianBlur(image, 3)
    median5 = cv2.medianBlur(image, 5)
    rank3 = rank_filter(image, 3, 2)
    rank5 = rank_filter(image, 5, 6)
    show_results([image, median3, median5, rank3, rank5],
                 ["Original", "Median 3 x 3", "Median 5 x 5",
                  "Rank 2 (3 x 3)", "Rank 6 (5 x 5)"], Path(name).stem + "_rank")


4. Use the thresholding tutorial provided on https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html. Perform both simple and adaptive thresholding. Compare both methods and discuss the tradeoffs between them. Make sure to include multiple images demonstrating the findings discussed in the writeup.
- Coins1
- Coins2
- screws

In [ ]:
# Locate utils whether the notebook opens from homework or the repository root.
from pathlib import Path
import sys

root = Path.cwd() if Path("images").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / "homework"))

from utils.ThresholdImages import ThresholdImages

segmentation = ThresholdImages().compare()



5. Use morphological operators to improve the segmentation of the images in problem 3. There are multiple operations with different masks and orders. Describe which set of operations works best and why. Make sure to include all images generated. See https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html for help on how to use
the operators.

In [ ]:
# Locate utils whether the notebook opens from homework or the repository root.
from pathlib import Path
import sys

root = Path.cwd() if Path("images").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / "homework"))

# Run question 4 first to create segmentation.
from utils.Morphology import Morphology

Morphology(segmentation).compare()
